# FleXgeo2 quickstart analysis

This notebook runs a first FleXgeo2 analysis on `pdb2lj5.pdb`, a 301-model NMR ensemble of human ubiquitin. The goal is to get from a PDB file to curvature/torsion descriptors, summary tables, and the standard overview plot with the smallest useful amount of code.

In [ ]:
from pathlib import Path
import sys
import tempfile


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "pdb2lj5.pdb").is_file():
            return candidate
    raise FileNotFoundError("Could not find the FleXgeo2 repo root containing pyproject.toml and pdb2lj5.pdb")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import matplotlib.image as mpimg
    import matplotlib.pyplot as plt
    import melodia_py  # noqa: F401
    import pandas as pd
    from flexgeo2 import AnalysisConfig, FlexGeo2App, OutputConfig
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing dependency: {exc.name}. Install the project in your notebook kernel with `pip install -e .` "
        "or, if you use uv, run `uv sync` and select the project environment as the Jupyter kernel."
    ) from exc

PDB_FILE = REPO_ROOT / "pdb2lj5.pdb"
tmpdir = tempfile.TemporaryDirectory(prefix="flexgeo2_quickstart_")
OUTPUT_DIR = Path(tmpdir.name)

print(f"Repo root: {REPO_ROOT}")
print(f"PDB file: {PDB_FILE}")
print(f"Temporary output directory: {OUTPUT_DIR}")

## Run the analysis

`FlexGeo2App` is the high-level library entrypoint. The config below keeps chain `A`, writes standard outputs to a temporary directory, and limits Melodia to one worker for predictable notebook behavior.

In [ ]:
config = AnalysisConfig(
    pdb_file=PDB_FILE,
    chains=["A"],
    n_jobs=1,
    output=OutputConfig(output_dir=OUTPUT_DIR, verbose=False, write_files=True),
)

result = FlexGeo2App().run(config)

## What came back?

The result object keeps the same information that FleXgeo2 writes to disk: raw per-model descriptors, per-residue ensemble summaries, and per-model summaries.

In [ ]:
model_count = result.raw_df["model"].nunique()
chain_count = result.raw_df["chain"].nunique()
residue_count = result.raw_df.groupby(["chain", "order"]).ngroups

print(f"Models: {model_count}")
print(f"Chains: {chain_count}")
print(f"Residues: {residue_count}")
print(f"Raw descriptor rows: {len(result.raw_df):,}")

In [ ]:
result.raw_df.head()

`raw_df` has one row per model, chain, and residue. The key descriptor columns are `curvature` and `torsion`.

In [ ]:
result.residue_summary_df.head()

`residue_summary_df` collapses the ensemble by residue, giving mean, standard deviation, minimum, and maximum values for curvature and torsion.

In [ ]:
result.overall_model_summary_df.head()

`overall_model_summary_df` summarizes each model and reports mean absolute deviation from the ensemble mean, which is useful for quickly finding conformations that differ most from the ensemble average.

In [ ]:
from dataclasses import fields

artifacts = result.outputs
for field in fields(artifacts):
    name = field.name
    path = getattr(artifacts, name)
    if path is not None:
        print(f"{name}: {path}")

## Overview plot

The overview plot shows the ensemble mean curvature and torsion for each residue, with a shaded standard deviation band.

In [ ]:
assert artifacts.overview_plot is not None and artifacts.overview_plot.is_file()
img = mpimg.imread(artifacts.overview_plot)
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis("off")
plt.show()